# 🎬 Movie Recommendation System

An interactive recommendation engine built with the **MovieLens 25M dataset**.  
Type a movie name → get a live search → get 10 personalised recommendations instantly.

**Dataset:** [MovieLens 25M](https://grouplens.org/datasets/movielens/25m/)  
**Techniques:** TF-IDF, Cosine Similarity, Collaborative Filtering  
**Tools:** pandas · scikit-learn · ipywidgets

## 1 · Load Movie Data

In [ ]:
import pandas as pd

# Dataset path — adjust if running from a different directory
movies = pd.read_csv("ml-25m/movies.csv")
print(f"Loaded {len(movies):,} movies")
movies.head()

## 2 · Clean Movie Titles with Regex

Strip everything that isn't a letter, digit, or space.  
This removes parentheses around years and special characters that would confuse the search engine.

In [ ]:
import re

def clean_title(title: str) -> str:
    """Remove all characters that are not letters, digits, or spaces."""
    return re.sub(r"[^a-zA-Z0-9 ]", "", title)

movies["clean_title"] = movies["title"].apply(clean_title)
movies.head()

## 3 · Build a TF-IDF Matrix

**TF-IDF** (Term Frequency – Inverse Document Frequency) converts text into numeric vectors.  
`ngram_range=(1,2)` means we also capture *pairs* of consecutive words (bigrams), making *Toy Story* a single searchable token.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf = vectorizer.fit_transform(movies["clean_title"])

print(f"TF-IDF matrix shape: {tfidf.shape}")

## 4 · Search Function (Cosine Similarity)

Cosine similarity measures the angle between two vectors — the closer to 1, the more similar.  
We transform the user's query into the same vector space and return the 5 closest titles.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def search(title: str) -> pd.DataFrame:
    """Return the 5 movies whose titles best match the query."""
    cleaned = clean_title(title)
    query_vec = vectorizer.transform([cleaned])
    similarity = cosine_similarity(query_vec, tfidf).flatten()
    indices = np.argpartition(similarity, -5)[-5:]
    results = movies.iloc[indices].iloc[::-1]
    return results[["movieId", "title", "genres"]]

## 5 · Interactive Movie Search Widget

Type any movie name below — results update in real time as you type.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value="Toy Story",
    description="🔍 Search:",
    layout=widgets.Layout(width="400px"),
    style={"description_width": "80px"},
)
movie_list = widgets.Output()

def on_type(change):
    with movie_list:
        movie_list.clear_output(wait=True)
        title = change["new"]
        if len(title) > 2:
            display(search(title))

movie_input.observe(on_type, names="value")
display(movie_input, movie_list)

## 6 · Load Ratings Data

25 million ratings from 162,541 users across 62,423 movies.  
Each row: a user rated a movie on a 0.5 – 5 ⭐ scale.

In [ ]:
ratings = pd.read_csv("ml-25m/ratings.csv")
print(f"Loaded {len(ratings):,} ratings")
ratings.head()

## 7 · Collaborative Filtering — Core Logic

**How it works:**
1. Find users who rated the target movie > 4 ⭐ (**similar users**)
2. Find every other movie those users also rated > 4 ⭐
3. Keep movies liked by > 10% of similar users (popularity threshold)
4. Compare against *all* users who liked those movies → compute a **score = similar% / all%**
5. High score = this movie is *specifically* popular with fans of your target, not just generally popular

Let's prototype with *The Avengers* (movieId 89745):

In [ ]:
movie_id = 89745  # The Avengers (2012)

# Step 1 – users who loved this movie
similar_users = ratings[
    (ratings["movieId"] == movie_id) & (ratings["rating"] > 4)
]["userId"].unique()
print(f"Users who loved this movie: {len(similar_users):,}")

In [ ]:
# Step 2 – other movies those users also loved
similar_user_recs = ratings[
    (ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)
]["movieId"]

# Step 3 – percentage of similar users who liked each movie (threshold > 10%)
similar_user_recs = similar_user_recs.value_counts() / len(similar_users)
similar_user_recs = similar_user_recs[similar_user_recs > 0.10]
print(f"Candidate movies after threshold: {len(similar_user_recs)}")

In [ ]:
# Step 4 – same movies among ALL users
all_users = ratings[
    (ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)
]
all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

# Step 5 – combine and score
rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
rec_percentages.columns = ["similar", "all"]
rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
rec_percentages = rec_percentages.sort_values("score", ascending=False)

rec_percentages.head(10).merge(movies, left_index=True, right_on="movieId")[
    ["score", "title", "genres"]
]

## 8 · Recommendation Function

In [ ]:
def find_similar_movies(movie_id: int) -> pd.DataFrame:
    """
    Given a movieId, return the top-10 recommended movies.
    
    Strategy: collaborative filtering — find users who loved this movie,
    then surface movies that are disproportionately popular with that group
    compared with the general user population.
    """
    similar_users = ratings[
        (ratings["movieId"] == movie_id) & (ratings["rating"] > 4)
    ]["userId"].unique()

    similar_user_recs = ratings[
        (ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)
    ]["movieId"]
    similar_user_recs = similar_user_recs.value_counts() / len(similar_users)
    similar_user_recs = similar_user_recs[similar_user_recs > 0.10]

    all_users = ratings[
        (ratings["movieId"].isin(similar_user_recs.index)) & (ratings["rating"] > 4)
    ]
    all_user_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

    rec_percentages = pd.concat([similar_user_recs, all_user_recs], axis=1)
    rec_percentages.columns = ["similar", "all"]
    rec_percentages["score"] = rec_percentages["similar"] / rec_percentages["all"]
    rec_percentages = rec_percentages.sort_values("score", ascending=False)

    return (
        rec_percentages.head(10)
        .merge(movies, left_index=True, right_on="movieId")[["score", "title", "genres"]]
        .reset_index(drop=True)
    )

# Quick test
find_similar_movies(89745)

## 9 · Full Interactive Recommendation Widget

Type a movie → the system finds it → returns 10 personalised recommendations.  
Everything updates live as you type.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

movie_name_input = widgets.Text(
    value="Toy Story",
    description="🎬 Movie:",
    layout=widgets.Layout(width="420px"),
    style={"description_width": "80px"},
)
recommendation_list = widgets.Output()

def on_type(change):
    with recommendation_list:
        recommendation_list.clear_output(wait=True)
        title = change["new"]
        if len(title) > 2:
            results = search(title)
            if not results.empty:
                movie_id = results.iloc[0]["movieId"]
                display(find_similar_movies(movie_id))

movie_name_input.observe(on_type, names="value")

print("Type a movie title below to get recommendations:")
display(movie_name_input, recommendation_list)

---
## ✅ Project Summary

| Step | What we built |
|------|---------------|
| Data cleaning | Regex to strip non-alphanumeric characters from titles |
| Search engine | TF-IDF + cosine similarity with bigrams |
| Recommendation engine | Collaborative filtering using 25M ratings |
| UI | Live interactive widgets via ipywidgets |

### Ideas to extend this project
- Add a **genre filter** as a second dropdown widget
- Incorporate **tags** from `tags.csv` for content-based filtering
- Use **genome scores** (`genome-scores.csv`) for tag-relevance weighting
- Deploy as a **Streamlit** or **Gradio** web app
